
# race critial perterson mutex test and set, compare and swaping, semeaphore

1. A university system generates student IDs using a shared variable next_id.
When multiple admission requests are processed simultaneously, some students receive duplicate IDs. No synchronization mechanism is currently used. Implement a C++ program demonstrating this issue

2. Two processes update a shared database record.
The system designer wants a software-only solution (no hardware support) that ensures:
    • Mutual exclusion 
    • Fair turn-taking 
Implement the solution in C++

3. A file system allows multiple threads to write logs.
To prevent corruption:
    • Only one thread should write at a time 
    • Others should wait efficiently (no busy spinning) 
Implement using C++

4. A bank has implemented an ATM system where multiple users can access their accounts simultaneously. Each transaction updates a shared variable called balance.
Recently, the bank noticed that sometimes the balance becomes incorrect when multiple users withdraw or deposit money at the same time.
The system designer decides to use a low-level hardware-supported locking mechanism where:
    • A lock variable is checked and set in one atomic step 
    • Threads repeatedly check the lock until it becomes free 
Tasks
    1. Identify the synchronization technique used 
    2. Write a C++ program to solve this problem

5. A multi-core system maintains a global counter that is updated by multiple threads. The system must ensure:
    • The counter is updated only if its current value has not been changed by another thread 
    • The update must be atomic 
The system uses a hardware instruction that:
    • Compares a variable with an expected value 
    • Updates it only if they match 
Tasks
    1. Identify the synchronization method 
    2. Implement the solution in C++

6. A computer lab has a single printer shared among many students.
Rules:
    • Only one student can use the printer at a time 
    • If the printer is busy, others must wait 
    • Waiting students should not waste CPU (no busy waiting) 
The system designer creates a high-level structure where:
    • Shared variables are hidden 
    • Only predefined functions can access the resource 
    • Waiting is handled automatically 
Tasks
    1. Identify the synchronization method 
    2. Write a C++ solution

7. A factory has:
    • Producers adding items 
    • Consumers removing items 
Rules:
    • Buffer size is limited 
    • Producers must wait if full 
    • Consumers must wait if empty 
Tasks
    1. Identify synchronization tool 
    2. Implement in C++


<br>

**Question 1:** A university system generates students IDs using a shared variable next_id. When multile admission requests are processed simultaneously, some students receive duplicated IDs. No synchronization mechanism is currently used .

<br>

In [24]:
import os 
import time 
import threading

#shared_variable
next_id = 5 

def assign_ids(std_name:str)->None:
    global next_id 
    
    current_id = next_id
    print("Student name: {} receive id: {}".format(std_name,current_id))
    
    # sleep: (in this time other thread can read this the shared variables)
    time.sleep(1)
    
    next_id = current_id+1
    print("Assiged Id: {} Student: {} Next ID: {}".format(current_id,std_name,next_id))
    
    
# create thread object in memory 
# must be add , at the end python will understand it's a tuple
t1 = threading.Thread(target=assign_ids,args=("Student-A",))
t2 = threading.Thread(target=assign_ids,args=("Student-B",))

# start the threads:
t1.start()
t2.start()

# wait until thread does not finished his execution:
t1.join()
t2.join()


Student name: Student-A receive id: 5
Student name: Student-B receive id: 5
Assiged Id: 5 Student: Student-A Next ID: 6
Assiged Id: 5 Student: Student-B Next ID: 6


In [ ]:


# ===== 1. Mutex: =====
next_id = 5 

# create a lock object
lock = threading.Lock()

def assign_ids(std_name:str)->None:
    global next_id
    
    # aquire
    try:
        lock.acquire()
        current_id = next_id
        print(f"Student name: {std_name} receive id: {current_id}")
        
        # wait that other's thread try to access the shared variables
        time.sleep(1)
        
        next_id = current_id+1
        print(f"Assiged Id: {current_id} Student: {std_name} Next ID: {next_id}")
    finally:
        lock.release()
    
t1 = threading.Thread(target=assign_ids,args=("Std_A",))
t2 = threading.Thread(target=assign_ids,args=("Std_B",))

t1.start()
t2.start()
t1.join()
t2.join()

Student name: Std_A receive id: 5
Assiged Id: 5 Student: Std_A Next ID: 6
Student name: Std_B receive id: 6
Assiged Id: 6 Student: Std_B Next ID: 7


In [ ]:


# === 1. Mutex problem with python context manager with ===
next_id = 5 
lock = threading.Lock()

def students_ids(std_name:str)->None:
    global next_id
    
    with lock:
        current = next_id
        print(f"{std_name} assign id: {current}")
        
        time.sleep(1)
        next_id = current+1 
        print(f"{std_name} assign id: {current} next_id: {next_id}")
        print("*"*30)
        

t1 = threading.Thread(target=students_ids,args=("STD_A",))
t2 = threading.Thread(target=students_ids,args=("STD_B",))

print("*"*30)

t1.start()
t2.start()
t1.join()
t2.join()



******************************
STD_A assign id: 5
STD_A assign id: 5 next_id: 6
******************************
STD_B assign id: 6
STD_B assign id: 6 next_id: 7
******************************


In [ ]:

# 2. === Test and Set ===
next_id = 5 
# mean no one in critical section
target = False

def test_and_set()->bool:
    global target
    r = target
    target = True
    return r 


def students_ids(std_name:str):
    global next_id 
    
    # === Entry section: text_and_set() ===
    while test_and_set():
        pass 
    
    
    # === critical section: ===
    try: 
        current = next_id
        print(f"{std_name} assign id: {current}")
        
        
        time.sleep(1)
        
        next_id = current+1 
        print(f"{std_name} assign id: {current} next_id: {next_id}")
        print("*"*30)
    
    # === exit section ===
    finally:
        global target
        target = False 
        
    
t1 = threading.Thread(target=students_ids,args=("STD_A",))
t2 = threading.Thread(target=students_ids,args=("STD_B",))

t1.start()
t2.start()
t1.join()
t2.join()



STD_A assign id: 5
STD_A assign id: 5 next_id: 6
******************************
STD_B assign id: 6
STD_B assign id: 6 next_id: 7
******************************


In [24]:
import time 
import threading

# ==== 3. Compare and swap ======
next_id = 5 
global value 
lock = [False]

# value = 0, cs is free else block
# expected_value = 0 for the process which want to entered in cs and new_value=1 
def compare_swap(value,excepted_value,new_value):
    temp =  value[0]
    if(value[0]==excepted_value):
        value[0] = new_value
    return temp 


def students_ids(std:str):
    global next_id
    
    # this is not cpp
    #lock[0] = False 
    while(compare_swap(lock,False,True)!=False):
        pass 
    
    current_id = next_id
    print(f"Std: {std} assign id: {current_id}")
    time.sleep(1)
    next_id  = current_id + 1 
    print(f"Std: {std} assign id: {current_id} next id is {next_id}")
    print("*"*40)
    lock[0] = False
    

t1 = threading.Thread(target=students_ids,args=("A",))
t2 = threading.Thread(target=students_ids,args=("B",))


t1.start()
t2.start()
    
t1.join()
t2.join()



Std: A assign id: 5
Std: A assign id: 5 next id is 6
****************************************
Std: B assign id: 6
Std: B assign id: 6 next id is 7
****************************************


In [ ]:


# === 4. Peterson solution: ===
turn = 0 
next_id = 5 
interested = [False, False]

def entry_section(process:int):
    global turn 
    global interested
    
    other = 1- process
    interested[process] = True 
    turn = other
    
    while (interested[other]==True and turn==other):
        pass 
    

def exit_section(process:int):
    global interested
    interested[process] = False
    

def students_ids(std_name:str,process_id:int):
    global next_id
    
    #entry section
    entry_section(process_id)
    
     # aquire
    try:
        current_id = next_id
        print(f"Student name: {std_name} receive id: {current_id}")
        
        # wait that other's thread try to access the shared variables
        time.sleep(1)
        
        next_id = current_id+1
        print(f"Assiged Id: {current_id} Student: {std_name} Next ID: {next_id}")
    finally:
        exit_section(process_id)
    
t1 = threading.Thread(target=students_ids,args=("STD1",0,))
t2 = threading.Thread(target=students_ids,args=("STD2",1,))


t1.start()
t2.start()

t1.join()
t2.join()


Student name: STD1 receive id: 5
Assiged Id: 5 Student: STD1 Next ID: 6
Student name: STD2 receive id: 6
Assiged Id: 6 Student: STD2 Next ID: 7


In [12]:
import time 
import threading

# === 5. Seamaphore solution: ===
next_id = 5 
s = [1]


def P(S:int):
    while(S[0]<=0):
        pass 
    S[0] = S[0] - 1 
    

def V(S:int):
    S[0] = S[0] + 1 


def students_ids(std:str):
    global next_id 
    P(s)
    
    current_id = next_id
    print(f"{std} is assign id: {current_id}")
    time.sleep(1)
    next_id  = current_id + 1 
    print(f"Assigned ID: {current_id} next id is: {next_id}")
    
    V(s)
    

t1 = threading.Thread(target=students_ids,args=("STD_A",))
t2 = threading.Thread(target=students_ids,args=("STD_B",))
    
t1.start()
t2.start()
t1.join()
t2.join()


STD_A is assign id: 5
Assigned ID: 5 next id is: 6
STD_B is assign id: 6
Assigned ID: 6 next id is: 7
